In [1]:
!nvidia-smi

Tue May 19 03:44:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -U langchain langchain-huggingface transformers accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: langgraph-checkpoint
    Found existing in

In [8]:
from google.colab import userdata
from huggingface_hub import login
import os

hf_token = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token

login(token=hf_token)

from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

pipeline_llm = HuggingFacePipeline.from_model_id(
    model_id="meta-llama/Llama-3.2-3B-Instruct",
    task="text-generation",
    model_kwargs={
        "token": hf_token,
        "device_map": "auto",
    },
    pipeline_kwargs={
        "max_new_tokens": 128,
        "temperature": 0.1,
        "top_p": 0.5,
        "do_sample": True,
        "return_full_text": False,
    }
)

chat_model = ChatHuggingFace(llm=pipeline_llm)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [9]:
result = chat_model.invoke("Hello! Reply in one short sentence.")
print(result.content)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


How can I assist you today?


In [10]:
from langchain_core.prompts import PromptTemplate

template = "What is the capital of {country}? Answer in one short sentence."
prompt = PromptTemplate.from_template(template)

print(prompt)

input_variables=['country'] input_types={} partial_variables={} template='What is the capital of {country}? Answer in one short sentence.'


In [11]:
from langchain_core.output_parsers import StrOutputParser

llm_chain = prompt | chat_model | StrOutputParser()

response = llm_chain.invoke({"country": "Korea"})
print(response)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The capital of South Korea is Seoul, while the capital of North Korea is Pyongyang.


In [12]:
# On math Problems

!pip install datasets

In [14]:
import torch
from huggingface_hub import login
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [15]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
model_name, device_map="cuda", torch_dtype=torch.float16
)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [16]:
from transformers import GenerationConfig
# 생성 관련 파라미터는 model.generation_config에 추가
model.generation_config = GenerationConfig(
max_new_tokens=128,
temperature=0.1,
top_p=0.9,
do_sample=True,
pad_token_id=tokenizer.eos_token_id,
)
# pipeline에는 pipeline 전용 인자만 남김
pipe = pipeline(
"text-generation",
model=model,
tokenizer=tokenizer,
return_full_text=False,
)
llm = HuggingFacePipeline(pipeline=pipe)

In [17]:
from datasets import load_dataset
dataset_math = load_dataset("cais/mmlu", "high_school_mathematics")

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

high_school_mathematics/test-00000-of-00(…):   0%|          | 0.00/33.7k [00:00<?, ?B/s]

high_school_mathematics/validation-00000(…):   0%|          | 0.00/6.99k [00:00<?, ?B/s]

high_school_mathematics/dev-00000-of-000(…):   0%|          | 0.00/4.50k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/270 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/29 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

In [18]:
print(dataset_math, "\n")
print("First training data:\n", dataset_math['dev'][0])

DatasetDict({
    test: Dataset({
        features: ['question', 'subject', 'choices', 'answer'],
        num_rows: 270
    })
    validation: Dataset({
        features: ['question', 'subject', 'choices', 'answer'],
        num_rows: 29
    })
    dev: Dataset({
        features: ['question', 'subject', 'choices', 'answer'],
        num_rows: 5
    })
}) 

First training data:
 {'question': 'Joe was in charge of lights for a dance. The red light blinks every two seconds, the yellow light every three seconds, and the blue light every five seconds. If we include the very beginning and very end of the dance, how many times during a seven minute dance will all the lights come on at the same time? (Assume that all three lights blink simultaneously at the very beginning of the dance.)', 'subject': 'high_school_mathematics', 'choices': ['3', '15', '6', '5'], 'answer': 1}


In [19]:
import pandas as pd

df_math_train = pd.DataFrame(dataset_math['dev'])
df_math_valid = pd.DataFrame(dataset_math['validation'])
df_math_test = pd.DataFrame(dataset_math['test'])

print(df_math_train)

                                            question                  subject  \
0  Joe was in charge of lights for a dance. The r...  high_school_mathematics   
1  Five thousand dollars compounded annually at a...  high_school_mathematics   
2  The variable $x$ varies directly as the square...  high_school_mathematics   
3  Simplify and write the result with a rational ...  high_school_mathematics   
4  Ten students take a biology test and receive t...  high_school_mathematics   

                                             choices  answer  
0                                      [3, 15, 6, 5]       1  
1                                     [12, 1, 30, 5]       2  
2             [-1, 16, -\frac{1}{256}, \frac{1}{16}]       2  
3  [\frac{3\sqrt{3}}{3}, \frac{1}{3}, \sqrt{3}, \...       3  
4                                   [55, 60, 62, 65]       3  


In [21]:
def get_options(df):
  options = []
  for _,row in df.iterrows():
    choices = row['choices'] # ['옵션A', '옵션B', '옵션C', '옵션D']
    opt_str =", ".join(f"{label}: {c}" for label, c in zip('ABCD', choices))
    options.append(opt_str)
  return options

options_train = get_options(df_math_train)
options_valid = get_options(df_math_valid)
options_test = get_options(df_math_test)
print(options_train)

['A: 3, B: 15, C: 6, D: 5', 'A: 12, B: 1, C: 30, D: 5', 'A: -1, B: 16, C: -\\frac{1}{256}, D: \\frac{1}{16}', 'A: \\frac{3\\sqrt{3}}{3}, B: \\frac{1}{3}, C: \\sqrt{3}, D: \\frac{\\sqrt{3}}{3}', 'A: 55, B: 60, C: 62, D: 65']


In [22]:
def get_inputs(df,options):
  return [{"query": df['question'][i], 'options': options[i]} for i in range(len(df))]

input_train = get_inputs(df_math_train, options_train)
input_valid = get_inputs(df_math_valid, options_valid)
input_test = get_inputs(df_math_test, options_test)

print(input_train[0])

{'query': 'Joe was in charge of lights for a dance. The red light blinks every two seconds, the yellow light every three seconds, and the blue light every five seconds. If we include the very beginning and very end of the dance, how many times during a seven minute dance will all the lights come on at the same time? (Assume that all three lights blink simultaneously at the very beginning of the dance.)', 'options': 'A: 3, B: 15, C: 6, D: 5'}


In [23]:
basic_template = """The following is multiple choice question.
Question: {query}
Options: {options}"""
basic_prompt = PromptTemplate.from_template(template=basic_template)
print(basic_prompt)


input_variables=['options', 'query'] input_types={} partial_variables={} template='The following is multiple choice question.\nQuestion: {query}\nOptions: {options}'


In [25]:
basic_chain = basic_prompt | llm | StrOutputParser()
# Invoke
response = basic_chain.invoke({
"query": dataset_math["validation"][0]["question"],
"options": options_valid[0]
})
print(response)
# Batch
responses = basic_chain.batch(input_valid[:5])
for i, res in enumerate(responses):
  print(f"********** {i + 1} **********\n", res.strip(), "\n")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



The best answer is C.


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

********** 1 **********
 The best answer is A. 

********** 2 **********
 , E: 8
The best answer is B.

## Step 1: Understand the problem
We are given a group of 11 people, with T of them always telling the truth and L of them always lying. Each person names two other people and claims that of the two, exactly one of them is lying.

## Step 2: Analyze the situation
We can start by considering the possible combinations of truth-tellers and liars. Let's assume there are T truth-tellers and L liars. Each person is named in this way by exactly two other people.

## Step 3: Consider the possible scenarios
There are several possible scenarios to consider. We can start by examining the case where T = 6, 7, or 8.

## Step 4: Analyze the case when T = 6
If T = 6, then there are 6 truth-tellers. We can analyze the situation by considering the number of liars. Since each person claims that of the two, exactly one of them is lying, there must be 5 liars.

## Step 5: Calculate the number of combina

In [27]:
!python --version

Python 3.12.13


In [28]:
!pip freeze > requirements.txt
!cat requirements.txt

absl-py==1.4.0
accelerate==1.13.0
access==1.1.10.post3
affine==2.4.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.5
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
alembic==1.18.4
altair==5.5.0
annotated-doc==0.0.4
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.13.0
anywidget==0.9.21
apsw==3.53.0.0
apswutils==0.1.2
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.3
arrow==1.4.0
arviz==0.22.0
astropy==7.2.0
astropy-iers-data==0.2026.4.20.0.58.15
astunparse==1.6.3
atpublic==5.1
attrs==26.1.0
audioread==3.1.0
Authlib==1.6.11
autograd==1.8.0
babel==2.18.0
backcall==0.2.0
beartype==0.22.9
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.39.0
bigquery-magics==0.14.0
bleach==6.3.0
blinker==1.9.0
blis==1.3.3
blobfile==3.2.0
blosc2==4.1.2
bokeh==3.8.2
Bottleneck==1.4.2
bqplot==0.12.45
branca==0.8.2
brotli==1.2.0
CacheControl==0.14.4
cachetools==6.2.6
catalogue==2.0.10
certifi==2026.4.22
